<table>
  <tr>
    <td><div align="left"><font size="30">Temperature control lab</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the Temperature Control Lab**

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()



# Introduction

The [Temperature Control Lab](https://apmonitor.com/heat.htm) has two heater/heatsink/sensor units.  Electrical power delivered to each unit is convereted to heat and dissipated by
convection to the ambient air and radiatively to the other unit.

<img src="https://apmonitor.com/pdc/uploads/Main/tclab_transparent.png" width="400">

# Theory

## A lumped model, and the Biot number

Each heater unit is a transistor heater and a temperature sensor on a small aluminium heatsink.  We model it as a single **lump** with one temperature, $T_1$ or $T_2$, the same throughout the unit.  That is only reasonable if heat spreads through the unit much faster than it leaves through its surface.  The **Biot number** measures this

$$Bi = \frac{h L_c}{k}$$

where $h$ is the heat transfer coefficient at the surface (W/m²/K), $k$ is the thermal conductivity of the solid (W/m/K), and $L_c = V/A$ is the volume of the body divided by its surface area, a characteristic length.  It is the ratio of the resistance to heat conduction *inside* the body, $L_c/k$, to the resistance to heat transfer at its *surface*, $1/h$.

When $Bi \ll 1$ (a rule of thumb is $Bi < 0.1$) the surface is the bottleneck: heat moves around inside the body so easily that the temperature differences within it are negligible, and the body can be described by one temperature obeying an ordinary differential equation rather than a temperature field obeying a partial differential equation.

For a unit of mass $m = 4$ g of aluminium ($\rho \approx 2700$ kg/m³, $k \approx 200$ W/m/K) the volume is $V \approx 1.5 \times 10^{-6}$ m³.  With a surface area $A = 10$ cm² this gives $L_c \approx 1.5$ mm, and with $h = U = 10$ W/m²/K

$$Bi \approx \frac{10 \times 1.5\times 10^{-3}}{200} \approx 10^{-4}$$

which is far below 0.1.  The unit is not a solid block of aluminium, the transistor and sensor are attached with imperfect thermal contact, so the temperature the sensor reads can lag that of the heatsink, but the lumped model is a good approximation.

## Heat transfer equations

Heat flows between a unit and its surroundings by convection and by radiation, and between the two units by conduction and by radiation.  In each case the transfer depends on the temperatures of the two sides, in kelvin.  $T_\infty$ is the ambient temperature.

**Convection to the air**

$$U A (T_\infty - T_i)$$

is Newton's law of cooling, with $A$ the surface area of the unit not facing the other unit.

**Radiation to the surroundings**

$$\epsilon \sigma A (T_\infty^4 - T_i^4)$$

where $\epsilon$ is the emissivity of the heatsink and $\sigma$ is the Stefan-Boltzmann constant.  Air is transparent to thermal radiation, so this is exchange with the walls and objects around the unit, which we take to be at $T_\infty$.

**Conduction between the units**

$$\dot{Q}_{C12} = U A_s (T_2 - T_1)$$

is the heat flow from unit 2 to unit 1 across the narrow gap and the shared mounting, where $A_s$ is the area between the heaters.  It looks like convection, but Fourier's law of conduction through a layer of thickness $d$ is $q = -kA\,dT/dx = (k/d)\,A\,(T_2 - T_1)$, which has exactly this form with a coefficient $k/d$.  A lumped model just needs one coefficient, here $U$, which stands for the conduction across the gap and any convection within it.  For air, $k \approx 0.026$ W/m/K, so a gap of 2.6 mm gives a coefficient of about 10 W/m²/K, the same size as $U$.

**Radiation between the units**

$$\dot{Q}_{R12} = \epsilon \sigma A_s (T_2^4 - T_1^4)$$

is the net radiation from the hotter unit to the colder one.  It uses the area $A_s$ between the heaters, and does not account for how much of each unit the other can see (the view factor), so it is an approximation.

$\dot{Q}_{C12}$ and $\dot{Q}_{R12}$ are positive when unit 2 is hotter than unit 1, so they heat unit 1 and cool unit 2.


## Dynamic Temperature Response Equations

The heat stored in a unit, $m c_p T_i$, changes at the rate at which heat flows in, so

$$m c_p \frac{dT_1}{dt} = U A (T_\infty - T_1) + \epsilon \sigma A (T_\infty^4 - T_1^4) + \dot{Q}_{C12} + \dot{Q}_{R12} + \alpha_1 u_1$$

$$m c_p \frac{dT_2}{dt} = U A (T_\infty - T_2) + \epsilon \sigma A (T_\infty^4 - T_2^4) - \dot{Q}_{C12} - \dot{Q}_{R12} + \alpha_2 u_2$$

where $m$ is the mass of the heater unit and $c_p$ is its specific heat capacity (mapping energy to temperature rise).  The terms are, in order, convection to the air, radiation to the surroundings, conduction to the other unit, radiation to the other unit, and the electrical heating.  $u_i$ is the heater setting in percent and $\alpha_i$ converts it to watts, the heater output $\dot{Q}_i = \alpha_i u_i$.

## Parameters

These are copied from the [online documentation](https://apmonitor.com/heat.htm)

| Quantity | Value |
| --- | --- |
| Initial temperature ($T_0$) | 296.15 K (23°C)
| Ambient temperature ($T_\infty$) | 296.15 K (23°C)
| Heater output ($\dot{Q}_1$) | 0 to 1 W
| Heater factor ($\alpha_1$) | 0.01 W/(% heater)
| Heater output ($\dot{Q}_2$) | 0 to 0.75 W
| Heater factor ($\alpha_2$) | 0.0075 W/(% heater)
| Heat capacity ($c_p$) | 500 J/kg/K
| Surface Area Not Between Heaters ($A$) | $1.0 \times 10^{-3} \text{ m}^2$ ($10 \text{ cm}^2$)
| Surface Area Between Heaters ($A_s$) | $2 \times 10^{-4} \text{ m}^2$ ($2 \text{ cm}^2$)
| Mass ($m$) | 0.004 kg (4 gm)
| Overall Heat Transfer Coefficient ($U$) | 10 W/m²/K
| Emissivity ($\epsilon$) | 0.9
| Stefan Boltzmann Constant ($\sigma$) | $5.67 \times 10^{-8} \text{ W/m}^2\text{/K}^4$

# Hardware

## About the Temperature Control Lab

This compact desktop laboratory experiment, shown above, comprises an Arduino Uno holding custom firmware, and a custom shield that holds the two heater units.  

## Firmware commands

The Arduino sketch can be found on [github](https://github.com/jckantor/TCLab-sketch/tree/master).
The firmware accepts simple ASCII command strings over USB-serial to set the heaters (in % of maximum) and to read the heater unit temperatures (in K).  

| Command | Argument | Description |
| --- | --- | --- |
| `A` | | software restart, returns "Start" |
| `LED` | float | set the LED brightness for 10 sec, range 0 to 100, returns the actual value |
| `P1` | float | set the PWM limit on heater 1, range 0 to 255, default 200, returns `P1` |
| `P2` | float | set the PWM limit on heater 2, range 0 to 255, default 100, returns `P2` |
| `Q1` | float | set heater 1, range 0 to 100, returns the value of `Q1` |
| `Q2` | float | set heater 2, range 0 to 100, returns the value of `Q2` |
| `R1` | | get the value of heater 1, range 0 to 100 |
| `R2` | | get the value of heater 2, range 0 to 100 |
| `SCAN` | | get the values of `T1`, `T2`, `Q1` and `Q2`, one per line |
| `T1` | | get the temperature of unit 1, returns `T1` in °C |
| `T2` | | get the temperature of unit 2, returns `T2` in °C |
| `VER` | | get the firmware version string |
| `X` | | stop and enter sleep mode, returns "Stop" |

It also implements safety logic and maximum temperature cutoffs.

# Implementation with bdsim

## Simulation

We start with the standard imports, create a bdsim simulation environment, and an empty block diagram to hold our model.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=0.02)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next, we define the parameters are Python variables.  The only change from the table above is that the heater factors $\alpha$ are scaled so that the heater commands are now in the range [0,1] rather than percentages in the range [0,100].

In [ ]:
T0 = 296.15       # Initial temperature (K)
T_inf = 296.15    # Ambient temperature (K)
Q1_max = 1.0      # Heater 1 maximum output (W)
α1 = 1            # Heater 1 factor (W/(heater))
Q2_max = 0.75     # Heater 2 maximum output (W)
α2 = 0.75         # Heater 2 factor (W/(heater))
Cp = 500.0        # Heat capacity (J/kg/K)
A = 1.0e-3        # Surface Area Not Between Heaters (m^2)
As = 2.0e-4       # Surface Area Between Heaters (m^2)
m = 0.004         # Mass (kg)
U = 10.0          # Overall Heat Transfer Coefficient (W/m^2/K)
ε = 0.9           # Emissivity
σ = 5.67e-8       # Stefan Boltzmann Constant (W/m^2/K^4)

K0 = 273.15       # 0C in Kelvin

We create two integrators whose states are the temperatures of two heater units.

In [ ]:
T1 = bd.INTEGRATOR(name="T1", x0=T0)  # Temperature of heater 1
T2 = bd.INTEGRATOR(name="T2", x0=T0)  # Temperature of heater 2

The inputs to the system are maximum value steps at 10s for heater 1 and 5 minutes for heater 2

In [ ]:
u1 = bd.STEP(T=10, on=Q1_max, name="heater1")   # Heater 1 input
u2 = bd.STEP(T=5*60, on=Q2_max, name="heater2")  # Heater 2 input

Next, we compute the heat transfer and the rates of change of temperature, which are assigned to the inputs of the respective integrators.

> **Note** that Python has no dot accent, so the variables `Q_C12` and `Q_R12` are the heat flows $\dot{Q}_{C12}$ and $\dot{Q}_{R12}$ (W) of the equations above.  `u1` and `u2`, defined above, are the heater commands $u_1$ and $u_2$ scaled to the range [0, 1], so `α1 * u1` is the heater output $\dot{Q}_1$ in watts, and `Q1_max` is its maximum.

In [ ]:
Q_C12 = U * As * (T2 - T1)  # Conduction heat transfer between heaters
Q_R12 = ε * σ * As * (T2**4 - T1**4)  # Radiation heat transfer between heaters

T1[0] = (U * A * (T_inf - T1) + ε * σ * A * (T_inf**4 - T1**4) + Q_C12 + Q_R12 + α1 * u1) / (m * Cp)
T2[0] = (U * A * (T_inf - T2) + ε * σ * A * (T_inf**4 - T2**4) - Q_C12 - Q_R12 + α2 * u2) / (m * Cp)

We setup a scope block to monitor the two temperatures in degrees Celsius

In [ ]:
T1C = T1 - K0  # Convert to Celsius
T2C = T2 - K0  # Convert to Celsius

bd.SCOPE(labels=("T_1", "T_2"),name="Temperature Scope", inputs=[T1C, T2C])  # display the temperatures on a scope

Our model is complete, so we now have to compile it.  This will flag any errors in the model  

In [ ]:
bd.compile()  # check the diagram
print(bd)
# bd.report_summary()

Now, we simulate the model for 10 minutes -- the system has very long time constants.  When we constructed the simulation runner at the top of this notebook, we set `animation_rate=0.02` which means that 1s of simulated model time is displayed in 20ms of clock time -- it displays 50x faster.

In [ ]:
out = sim.run(bd, T=15*60, watch=[u1, u2, T1, T2])  # simulate for 15min
print(out)

We can see the initial rapid rise of unit 1 once it is activated at $t=5$s and the coupling of some heat to unit 2. When unit 2's heater is activated at $t=300$s we see its temperature rise rapidly, but the effect on unit 1 is imperceptible.  The heater in unit 2 is only 75% that of unit 1, so it assymptotes to a lower temperature than unit 1.

## Linearization

In [ ]:
lin = bd.linearize(full=True)
print(lin)

In [ ]:
lin50 = bd.linearize(x=[50+K0, 50+K0], full=True)
lin50.ss()

## Control

Let's build an LQR controller to regulate the temperature of the two heaters to follow individual triangle wave setpoints (180°) phase difference varying between 35C and 55C (mean 45C).

We start by linearizing the system about the mean

In [ ]:
lin = bd.linearize(x=[45+K0, 45+K0], full=True)

In [ ]:
import control as ct

lin.ss()

In [ ]:
import control as ct

# Q must be positive semi-definite (penalizes state errors)
# R must be positive definite (penalizes control effort)
Q = np.diag([1, 1])  # weights for [T1, T2]
R = np.array([[1, 1]]) # weight for control inputs (u1, u2)

# 3. Compute the LQR optimal gain matrix K control.lqr returns:
# K: The optimal feedback gain matrix (u = -Kx)
# S: The solution to the algebraic Riccati equation
# E: The eigenvalues (poles) of the closed-loop system (A - BK)
K, S, E = ct.lqr(lin.ss, Q, R)